<a href="https://colab.research.google.com/github/wasikhan2004th-cpu/khan/blob/main/explainable_deepfake_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explainable Deepfake Face Detection Using EfficientNet-B0 and Vision Transformer with Grad-CAM

This notebook provides a complete, production-grade pipeline for deepfake face detection using a hybrid Deep Learning architecture that fuses local convolutional features (**EfficientNet-B0**) with global context relationships (**Vision Transformer / ViT**).

### Features Included:
1. **Automated Kaggle Dataset Download & Unzipping** (`xhlulu/140k-real-and-fake-faces`)
2. **Strict Dataset Balancing & Pipeline Optimization** using `torch.utils.data`
3. **Hybrid Architecture**: EfficientNet-B0 CNN feature extractor coupled to a custom multi-head Attention/Transformer block
4. **Production Training Setup**: Mixed Precision (AMP), Checkpointing, and Early Stopping
5. **Comprehensive Evaluation**: Confusion Matrix, ROC-AUC, Precision, Recall, and F1-Score
6. **Explainability**: Custom Grad-CAM implementation targeting the final convolutional layers of the EfficientNet backbone
7. **Interactive Deployment**: Seamlessly integrated Streamlit application code, ready to be hosted via LocalTunnel directly from Colab.

## Step 1: Environment Setup & Package Installation

In [ ]:
# Install necessary packages for modern deep learning, evaluation, and deployment
!pip install -q timm albumentations streamlit pyngrok scikit-learn matplotlib seaborn opencv-python-headless

## Step 2: Kaggle Credentials & Automated Dataset Download
To seamlessly download the dataset, please upload your `kaggle.json` API token file or input your Kaggle username and API key below.

In [ ]:
import os
import json
from google.colab import files

# Check if kaggle.json is already present, if not prompt upload
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json API token.")
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(uploaded[fn])
        os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download the 140k Real and Fake Faces Dataset from Kaggle
print("Downloading xhlulu/140k-real-and-fake-faces dataset...")
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p /content/dataset
print("Dataset unzipped successfully at /content/dataset")

## Step 3: Library Imports & Configuration

In [ ]:
import os
import glob
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

# Set seeds for complete reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Global Configurations
CONFIG = {
    'image_size': 224,
    'batch_size': 64,
    'epochs': 15,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'model_path': 'best_hybrid_deepfake_model.pth',
    'patience': 3,
    'samples_per_class': 10000 # Sampled subset to ensure fast and perfectly balanced pipeline training
}

## Step 4: Data Preparation & Exact Dataset Balancing

In [ ]:
def create_balanced_dataframe(base_dir, max_samples):
    splits = ['train', 'valid', 'test']
    dfs = {}

    for split in splits:
        real_paths = glob.glob(os.path.join(base_dir, split, 'real', '*.jpg')) + glob.glob(os.path.join(base_dir, split, 'real', '*.png'))
        fake_paths = glob.glob(os.path.join(base_dir, split, 'fake', '*.jpg')) + glob.glob(os.path.join(base_dir, split, 'fake', '*.png'))

        # Strict Balancing limit per split
        limit = max_samples if split == 'train' else int(max_samples * 0.2)
        limit = min(limit, len(real_paths), len(fake_paths))

        random.shuffle(real_paths)
        random.shuffle(fake_paths)

        real_selected = real_paths[:limit]
        fake_selected = fake_paths[:limit]

        paths = real_selected + fake_selected
        labels = [0] * len(real_selected) + [1] * len(fake_selected)

        df = pd.DataFrame({'image_path': paths, 'label': labels})
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        dfs[split] = df
        print(f"{split.capitalize()} split created: {len(real_selected)} Real, {len(fake_selected)} Fake. Total: {len(df)}")

    return dfs['train'], dfs['valid'], dfs['test']

DATASET_BASE_DIR = '/content/dataset/real_and_fake/real_and_fake_14k' if os.path.exists('/content/dataset/real_and_fake/real_and_fake_14k') else '/content/dataset'
train_df, valid_df, test_df = create_balanced_dataframe(DATASET_BASE_DIR, CONFIG['samples_per_class'])

## Step 5: PyTorch Custom Dataset & Image Augmentation Pipeline

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['label']

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.float32)

# Data Augmentation Pipelines
train_transforms = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transforms = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Generate DataLoaders
train_dataset = DeepfakeDataset(train_df, transform=train_transforms)
valid_dataset = DeepfakeDataset(valid_df, transform=val_test_transforms)
test_dataset = DeepfakeDataset(test_df, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

## Step 6: Hybrid Deep Learning Architecture (EfficientNet-B0 + ViT)

In [ ]:
class HybridEfficientNetViT(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4, transformer_layers=2):
        super(HybridEfficientNetViT, self).__init__()

        # EfficientNet-B0 Backbone Feature Extractor
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, features_only=True, out_indices=[4])
        # Feature dimension of EfficientNet-B0 stage 4 is 1280
        self.cnn_out_dim = 1280

        # Dimension reduction to map to Transformer Context Window
        self.proj = nn.Conv2d(self.cnn_out_dim, embed_dim, kernel_size=1)

        # Positional Embeddings for Transformer Processing over CNN Grids
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, 7 * 7 + 1, embed_dim))

        # Vision Transformer Encoder layers
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4, dropout=0.1, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)

        # Classification Head
        self.mlp_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        # Extraction
        features = self.backbone(x)[0] # Shape: [Batch, 1280, 7, 7]

        # Map channels to Embedding dimensions
        projected = self.proj(features) # Shape: [Batch, embed_dim, 7, 7]

        # Flatten map layout into Sequence patches
        batch_size, C, H, W = projected.shape
        patches = projected.flatten(2).transpose(1, 2) # Shape: [Batch, 49, embed_dim]

        # Prepend Class Token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, patches), dim=1) # Shape: [Batch, 50, embed_dim]
        x = x + self.pos_embedding

        # Process global contexts via Vision Transformer Sequence Encoder
        transformer_out = self.transformer(x)

        # Predict based on aggregated class sequence representations
        out_token = transformer_out[:, 0]
        logits = self.mlp_head(out_token)
        return logits.squeeze(1)

model = HybridEfficientNetViT().to(CONFIG['device'])

## Step 7: Optimized Model Training with AMP, Checkpointing, and Early Stopping

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
scaler = GradScaler() # Automated Mixed Precision

best_val_loss = float('inf')
patience_counter = 0

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("Starting Hybrid Network Training Engine Pipeline...")
for epoch in range(CONFIG['epochs']):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    # Validation Engine Block
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            preds = torch.round(torch.sigmoid(outputs))
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_val_loss = running_val_loss / len(valid_loader.dataset)
    epoch_val_acc = correct / total
    scheduler.step()

    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] - Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # Model Checkpoint and Early Stopping Evaluator
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), CONFIG['model_path'])
        print(f"==> Checkpoint Saved: Improved Validation Loss to {best_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"Early Stopping Triggered after {CONFIG['patience']} epochs without loss improvement.")
            break

## Step 8: Multi-Metric Model Evaluation Dashboard

In [ ]:
# Load best checkpoint weight layout
model.load_state_dict(torch.load(CONFIG['model_path']))
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(CONFIG['device'])
        with autocast():
            outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = np.round(probs)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
roc_auc = roc_auc_score(all_labels, all_probs)

print(f"\n--- Detailed Test Set Metrics Evaluation ---")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}\n")

# Plots Generation
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix Layout Map
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=ax[0])
ax[0].set_title('Confusion Matrix Matrix Configuration')
ax[0].set_ylabel('True Label')
ax[0].set_xlabel('Predicted Label')

# 2. ROC-AUC Curves Curve Layout
fpr, tpr, _ = roc_curve(all_labels, all_probs)
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (Area = {roc_auc:.4f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('Receiver Operating Characteristic')
ax[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

## Step 9: Custom Explainable AI Visualizer (Grad-CAM)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate_cam(self, input_tensor, target_category=None):
        self.model.eval()
        output = self.model(input_tensor)

        score = torch.sigmoid(output)
        self.model.zero_grad()
        output.backward(retain_graph=True)

        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (CONFIG['image_size'], CONFIG['image_size']))
        cam = cam - np.min(cam)
        cam = cam / (np.max(cam) + 1e-8)
        return cam, score.item()

def plot_gradcam(image_path, model):
    target_layer = model.backbone.blocks[-1][-1]
    cam_engine = GradCAM(model, target_layer)

    orig_image = cv2.imread(image_path)
    orig_image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
    orig_image = cv2.resize(orig_image, (CONFIG['image_size'], CONFIG['image_size']))

    input_tensor = val_test_transforms(image=orig_image)['image'].unsqueeze(0).to(CONFIG['device'])

    mask, prob = cam_engine.generate_cam(input_tensor)
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlaid = cv2.addWeighted(np.uint8(orig_image), 0.6, np.uint8(heatmap), 0.4, 0)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(orig_image)
    ax[0].set_title("Original Image View")
    ax[0].axis('off')

    ax[1].imshow(mask, cmap='jet')
    ax[1].set_title(f"Grad-CAM Heatmap Probability Layout")
    ax[1].axis('off')

    ax[2].imshow(overlaid)
    ax[2].set_title(f"Blended Visual Prediction score: {prob:.4f} ({'Fake' if prob >= 0.5 else 'Real'})")
    ax[2].axis('off')
    plt.show()

sample_row = test_df.sample(n=1).iloc[0]
plot_gradcam(sample_row['image_path'], model)

## Step 10: Production Streamlit Dashboard Deployment Script Setup
We write out an completely self-contained Python deployment file layout structure executable anywhere.

In [ ]:
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import cv2
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

st.set_page_config(page_title="XAI Deepfake Detection Engine Dashboard Panel", layout="wide")
st.title("Explainable Deepfake Face Detection Engine Portal")
st.write("Fusing Localized Convolutional Channels (EfficientNet-B0) with Sequential Multi-Head Self-Attention layers (ViT) accompanied with real-time Grad-CAM explaining capabilities.")

class HybridEfficientNetViT(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4, transformer_layers=2):
        super(HybridEfficientNetViT, self).__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=False, features_only=True, out_indices=[4])
        self.proj = nn.Conv2d(1280, embed_dim, kernel_size=1)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, 7 * 7 + 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
        self.mlp_head = nn.Sequential(nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x):
        features = self.backbone(x)[0]
        projected = self.proj(features)
        batch_size, C, H, W = projected.shape
        patches = projected.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, patches), dim=1) + self.pos_embedding
        transformer_out = self.transformer(x)
        return self.mlp_head(transformer_out[:, 0]).squeeze(1)

@st.cache_resource
def load_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = HybridEfficientNetViT()
    model.load_state_dict(torch.load('best_hybrid_deepfake_model.pth', map_location=device))
    model.to(device)
    model.eval()
    return model, device

try:
    model, device = load_model()
    st.success("Hybrid Model Framework checkpoints fully operational and loaded successfully!")
except Exception as e:
    st.error(f"Error fetching system checkpoints: {e}.")

uploaded_file = st.file_uploader("Provide validation sample image face path layout layer target file:", type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.image(image, caption="Source Target Input Stream View", use_container_width=True)

    transforms = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

    resized_image = cv2.resize(image, (224, 224))
    input_tensor = transforms(image=resized_image)['image'].unsqueeze(0).to(device)

    grad_activated, grad_gradients = [], []
    def f_hook(module, input, output): grad_activated.append(output)
    def b_hook(module, g_in, g_out): grad_gradients.append(g_out[0])

    t_layer = model.backbone.blocks[-1][-1]
    h1 = t_layer.register_forward_hook(f_hook)
    h2 = t_layer.register_full_backward_hook(b_hook)

    outputs = model(input_tensor)
    prob = torch.sigmoid(outputs).item()
    model.zero_grad()
    outputs.backward()

    act = grad_activated[0].cpu().data.numpy()[0]
    grad = grad_gradients[0].cpu().data.numpy()[0]
    weights = np.mean(grad, axis=(1, 2))

    cam = np.zeros(act.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights): cam += w * act[i]
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224, 224))
    cam = (cam - np.min(cam)) / (np.max(cam) + 1e-8)

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlaid = cv2.addWeighted(np.uint8(resized_image), 0.6, np.uint8(heatmap), 0.4, 0)

    h1.remove(); h2.remove()

    with col2:
        st.image(cam, caption="Computed Spatial Heatmap Density Layout", use_container_width=True, clamp=True)
    with col3:
        st.image(overlaid, caption="Grad-CAM Overlay Interpretation View Mapping Map", use_container_width=True)

    st.metric(label="Deepfake Probability Scoring Indicator", value=f"{prob*100:.2f}%")
    if prob >= 0.5:
        st.error("Classification Verdict Warning Notice Flagged: FAKE / MANIPULATED SIGNATURE DETECTED")
    else:
        st.success("Classification Verdict Safe Notice Flagged: REAL FACE VERIFIED")
''')
print('Streamlit Web Server Dashboard Component Script Generated successfully as app.py.')

## Step 11: Launch the Application (Run directly inside Google Colab via LocalTunnel)
Execute the cell block below, follow the printed URL endpoint hyperlink configurations to interact directly with your deployment.

In [ ]:
print("Your LocalTunnel Password Authorization Gateway Token Key is:")
!curl ipv4.icanhazip.com

!nohup streamlit run app.py --server.port 8501 &

!npx localtunnel --port 8501